In [3]:
import sys
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset, Subset
from torchvision.utils import make_grid
import torch.nn.functional as F


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import random

vae_path = "/home/qiyuanliu/data_filter/Verified-Synthetic-Data/MNIST/conv_cvae"
sys.path.append(vae_path)

import models as models
import train_helper as train_helper
import utils as utils
import data_helper as data_helper

# Set up device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_seed = 0
torch.manual_seed(base_seed)
torch.cuda.manual_seed_all(base_seed)
np.random.seed(base_seed)
random.seed(base_seed)

ROOT = "/home/qiyuanliu/data_filter/Verified-Synthetic-Data/MNIST/conv_cvae/schedule_size_no_filter"
model_saved_path = os.path.join(ROOT,"model_saved_no_filter_more")
data_saved_path = os.path.join(ROOT,"data_saved_no_filter_more")
results_saved_path = os.path.join(ROOT,"results_saved_no_filter_more")
picture_saved_path = os.path.join(ROOT,"picture_saved_no_filter_more")
os.makedirs(results_saved_path, exist_ok=True)
############## size schedule #################

k = 40
size_schedule = [32_000, 128_000, 256_000] + [512_000] * (k - 3)
#size_schedule = [512_000] * k

############################

full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())

test_dataset = datasets.MNIST(root="./data", train=False, download=True,transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
full_digit_indices = utils.create_balanced_subset_indices(full_dataset,seed=base_seed)

import torch
import torch.nn.functional as F

@torch.no_grad()
def generate_images_in_batches(model, total_samples, latent_dim, num_classes, batch_size=10000, device='cuda'):
    model.eval()
    generated_images = []
    all_labels = []

    # Balanced label assignment (always length = total_samples)
    labels_full = torch.arange(total_samples) % num_classes

    for start in range(0, total_samples, batch_size):
        end = min(start + batch_size, total_samples)
        n = end - start

        # Sample latent z
        z = torch.randn(n, latent_dim, device=device)

        # Labels
        y = labels_full[start:end]
        y_onehot = F.one_hot(y, num_classes=num_classes).float().to(device)

        # Decode logits → sigmoid → [0,1]
        logits_flat = model.decoder.decode(z, y_onehot)     # (n, 784), logits
        imgs = torch.sigmoid(logits_flat).view(-1, 1, 28, 28).cpu()

        generated_images.append(imgs)
        all_labels.append(y)

    images = torch.cat(generated_images, dim=0)
    labels = torch.cat(all_labels, dim=0)
    return images, labels

from FID import calculate_fid_score

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

def append_result(csv_path, model_name, val_loss, val_recon, val_kl, fid_score):
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    row = pd.DataFrame([{
        "model_name": model_name,
        "val_loss": float(val_loss),
        "val_recon": float(val_recon),
        "val_kl": float(val_kl),
        "fid": float(fid_score),
    }])
    header_needed = not os.path.exists(csv_path)  # only first write has header
    row.to_csv(csv_path, mode="a", header=header_needed, index=False)



@torch.no_grad()
def plot_model_samples(model, save_path=None, latent_dim=20, num_classes=10, per_class=8, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()

    # latent + labels
    z = torch.randn(num_classes * per_class, latent_dim, device=device)
    y = torch.arange(num_classes, device=device).repeat_interleave(per_class)
    y_onehot = F.one_hot(y, num_classes=num_classes).float().to(device)

    # decode -> logits, then map to [0,1]
    logits_flat = model.decoder.decode(z, y_onehot)            # (n, 784), logits
    imgs = torch.sigmoid(logits_flat).view(-1, 1, 28, 28)      # tensor on device, no grad (due to @no_grad)

    # move for plotting
    imgs_np = imgs.detach().cpu().numpy()

    fig, axes = plt.subplots(num_classes, per_class, figsize=(2*per_class, 2*num_classes))
    for c in range(num_classes):
        for j in range(per_class):
            idx = c * per_class + j
            axes[c, j].imshow(imgs_np[idx].squeeze(), cmap='gray')
            axes[c, j].axis('off')
            if j == 0:
                axes[c, j].set_ylabel(f"Class {c}", fontsize=10)

    plt.tight_layout()
    if save_path:
        if not os.path.exists(os.path.dirname(save_path)):
            os.makedirs(os.path.dirname(save_path))
        plt.savefig(save_path, dpi=150)
        print(f"[SAVE] Sample grid saved -> {save_path}")
    return fig, axes


def fid(model):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    synthetic_gen_size = 6000
    gen_imgs_before_filter,y_before_filter = generate_images_in_batches(
        model=model,
        total_samples=synthetic_gen_size,
        latent_dim=20,
        num_classes=10,
        batch_size=10000,
        device=device
    )
    # Load synthetic data
    #synthetic = torch.load(f"data_saved/synthetic_mnist_cvae_{sample_size}_2.pt")
    images = gen_imgs_before_filter # [N, 1, 28, 28]
    labels = y_before_filter  # [N]

    transform = transforms.ToTensor()

    real_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    synthetic_ds = TensorDataset(images, labels)
    
    synthetic_ds = TensorDataset(images, labels)
    fid = calculate_fid_score(real_ds, synthetic_ds)
    
    return fid

import shutil

init_size = 500

all_models = []
test_results = {"val_loss":[], "val_recon":[], "val_kl":[], "fid":[],"model_name":[]}

# Seed real subset & train initial model
init_subset = utils.get_balanced_subset(full_digit_indices, init_size)
init_dataset = Subset(full_dataset, init_subset)
init_loader = DataLoader(init_dataset, batch_size=128, shuffle=True)

this_model = models.CVAE(input_dim=784, label_dim=10, latent_dim=20,
                         name=f"cvae_conv_real_{init_size}", arch="conv").to(device)
model_saved_path = "/home/qiyuanliu/data_filter/Verified-Synthetic-Data/MNIST/conv_cvae/model_saved"
ckpt_path = os.path.join(model_saved_path, "cvae_conv_real_500.pth")
this_model.load_state_dict(torch.load(ckpt_path, map_location=device))

full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root="./data", train=False, download=True,transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

curr_model = this_model
discriminator_dataset = data_helper.prepare_discriminator_dataset(full_dataset, curr_model, device)
disc_loader = DataLoader(discriminator_dataset, batch_size=128, shuffle=True)
disc_model = models.SyntheticDiscriminator(input_dim=784).to(device)
train_helper.train_model(model=disc_model, train_loader=disc_loader, device=device,
                            epochs=80, lr=1e-3, patience=5, verbose=True)

/tmp/ipykernel_219854/876103063.py:185: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  this_model.load_state_dict(torch.load(ckpt_path, map_location=device))


Epoch [1/80] completed. Train Avg Loss: 0.2042, accuracy: 0.9107
Epoch [2/80] completed. Train Avg Loss: 0.0788, accuracy: 0.9698
Epoch [3/80] completed. Train Avg Loss: 0.0510, accuracy: 0.9808
Epoch [4/80] completed. Train Avg Loss: 0.0377, accuracy: 0.9855
Epoch [5/80] completed. Train Avg Loss: 0.0291, accuracy: 0.9891
Epoch [6/80] completed. Train Avg Loss: 0.0239, accuracy: 0.9915
Epoch [7/80] completed. Train Avg Loss: 0.0207, accuracy: 0.9923
Epoch [8/80] completed. Train Avg Loss: 0.0188, accuracy: 0.9930
Epoch [9/80] completed. Train Avg Loss: 0.0167, accuracy: 0.9938
Epoch [10/80] completed. Train Avg Loss: 0.0132, accuracy: 0.9949
Epoch [11/80] completed. Train Avg Loss: 0.0128, accuracy: 0.9953
Epoch [12/80] completed. Train Avg Loss: 0.0109, accuracy: 0.9960
Epoch [13/80] completed. Train Avg Loss: 0.0110, accuracy: 0.9961
EarlyStopping counter: 1 out of 5
Epoch [14/80] completed. Train Avg Loss: 0.0099, accuracy: 0.9965
Epoch [15/80] completed. Train Avg Loss: 0.0104, ac

{'train_losses': [0.20422787180741628,
  0.07881187796493372,
  0.05100964672168096,
  0.03769109722549717,
  0.029106697065755725,
  0.02389890574707339,
  0.020679194298029568,
  0.018781734932555506,
  0.01672019963938898,
  0.013193592908900853,
  0.012807763706225281,
  0.01085340336014633,
  0.010959927044735135,
  0.009907240838755388,
  0.010360336073120318,
  0.008687675464015531,
  0.009391588914345387,
  0.007169239272736498,
  0.008254815521060178,
  0.007192916322870103,
  0.005657660035351971,
  0.006976696763765843,
  0.007110162679539644,
  0.005005947963786457,
  0.0062085842480579835,
  0.005031295806446239,
  0.005200572588902165,
  0.004329176581965718,
  0.0054970259182448595,
  0.004104962587998367,
  0.004696905882173102,
  0.004356038968844716,
  0.003951196065996798,
  0.004252166649654464,
  0.003868936857915196,
  0.004010587593027094,
  0.00365703104780805,
  0.003844277668376647,
  0.0033937634219788834,
  0.0028057590796014666,
  0.0034725212141841137,
  0

In [ ]:
import sys
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset, Subset
from torchvision.utils import make_grid
import torch.nn.functional as F
import os
import argparse

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import random

vae_path = "/home/qiyuanliu/data_filter/Verified-Synthetic-Data/MNIST/conv_cvae"
sys.path.append(vae_path)

import models as models
import train_helper as train_helper
import utils as utils
import data_helper as data_helper


# Set up device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_seed = 0
torch.manual_seed(base_seed)
torch.cuda.manual_seed_all(base_seed)
np.random.seed(base_seed)
random.seed(base_seed)

full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())

test_dataset = datasets.MNIST(root="./data", train=False, download=True,transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
full_digit_indices = utils.create_balanced_subset_indices(full_dataset,seed=base_seed)

#train_dataset_5000 = Subset(full_dataset, range(5000))
import torch
import torch.nn.functional as F
init_size = 5000

all_models = []
test_results = {"val_loss":[], "val_recon":[], "val_kl":[], "fid":[],"model_name":[]}

# Seed real subset & train initial model
init_subset = utils.get_balanced_subset(full_digit_indices, init_size)
init_dataset = Subset(full_dataset, init_subset)
init_loader = DataLoader(init_dataset, batch_size=128, shuffle=True)

this_model = models.CVAE(input_dim=784, label_dim=10, latent_dim=20,
                         name=f"cvae_conv_real_{init_size}", arch="conv").to(device)
train_helper.train_model(this_model, init_loader, device, epochs=200, lr=1e-3, patience=5, verbose=False)

full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root="./data", train=False, download=True,transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

curr_model = this_model
discriminator_dataset = data_helper.prepare_discriminator_dataset(full_dataset, curr_model, device)
disc_loader = DataLoader(discriminator_dataset, batch_size=128, shuffle=True)
disc_model = models.SyntheticDiscriminator(input_dim=784).to(device)
val_dataset = data_helper.prepare_discriminator_dataset(test_dataset, curr_model, device)
train_helper.train_model_with_validation(model=disc_model, train_loader=disc_loader, val_loader=val_dataset, device=device,
                            epochs=80, lr=1e-3, patience=5, verbose=True)

RuntimeError: Class values must be smaller than num_classes.

In [4]:
curr_model = this_model
discriminator_dataset = data_helper.prepare_discriminator_dataset(full_dataset, curr_model, device)
disc_loader = DataLoader(discriminator_dataset, batch_size=128, shuffle=True)
disc_model = models.SyntheticDiscriminator(input_dim=784).to(device)
val_dataset = data_helper.prepare_discriminator_dataset(test_dataset, curr_model, device)
disc_loader_val = DataLoader(val_dataset, batch_size=128, shuffle=True)
train_helper.train_model_with_validation(model=disc_model, train_loader=disc_loader, val_loader=disc_loader_val, device=device,
                            epochs=80, lr=1e-3, patience=5, verbose=True)

Epoch [1/80] completed. Train Avg Loss: 0.5375, Val Avg Loss: 0.4442
Train Summary: accuracy: 0.7166, Val Summary: accuracy: 0.7923
Epoch [2/80] completed. Train Avg Loss: 0.3642, Val Avg Loss: 0.3280
Train Summary: accuracy: 0.8369, Val Summary: accuracy: 0.8585
Epoch [3/80] completed. Train Avg Loss: 0.2617, Val Avg Loss: 0.2982
Train Summary: accuracy: 0.8885, Val Summary: accuracy: 0.8739
Epoch [4/80] completed. Train Avg Loss: 0.1883, Val Avg Loss: 0.2997
Train Summary: accuracy: 0.9237, Val Summary: accuracy: 0.8774
Epoch [5/80] completed. Train Avg Loss: 0.1450, Val Avg Loss: 0.2689
Train Summary: accuracy: 0.9416, Val Summary: accuracy: 0.8978
Epoch [6/80] completed. Train Avg Loss: 0.1131, Val Avg Loss: 0.2066
Train Summary: accuracy: 0.9553, Val Summary: accuracy: 0.9224
Epoch [7/80] completed. Train Avg Loss: 0.0978, Val Avg Loss: 0.1938
Train Summary: accuracy: 0.9618, Val Summary: accuracy: 0.9258
Epoch [8/80] completed. Train Avg Loss: 0.0846, Val Avg Loss: 0.3516
Train S

{'train_losses': [0.5375221081415812,
  0.3641605312983195,
  0.2616612313191096,
  0.18827761390209197,
  0.1449962691028913,
  0.1130637631336848,
  0.09777953706582387,
  0.08464078299403191,
  0.07303949087560177,
  0.06547082067330678,
  0.062458489376058184,
  0.053494862012565134,
  0.04941384498924017,
  0.047909601769347984,
  0.04485209315791726,
  0.041540566954016686,
  0.037224503175293404,
  0.03935285229757428,
  0.03599271586065491,
  0.034253381221989794,
  0.03232058175262064,
  0.030321172112599014,
  0.030682019372905295,
  0.03047091040275991,
  0.02504792195657889,
  0.027299745484876136,
  0.024635404096512745,
  0.024906816805340348,
  0.02247140241560216,
  0.023064873770345003,
  0.023434039302465196,
  0.022418219543620943,
  0.018534376427189758,
  0.021211358891241254,
  0.019169472943634417,
  0.019101797517032053,
  0.020268836792030683,
  0.017083597365161404,
  0.019052411703454952,
  0.015595575998490676,
  0.01642133406656794,
  0.016958055954364438,
